#### Основные принципы реализации этого проекта
1. Радиусы соответствуют идеальным зонам Фурье
2. Коэффициенты кривой основаны на пред-тренированной модели
3. Цикл оптимизации будет подбирать коэффициенты, а также возможно и радиусы

## STEP 1
Настройка проекта

In [38]:
import numpy as np
import matplotlib.pyplot as plt
import meep as mp
import conf
from lib import sourses


"""
Преднастройки
TODO: перенести в конфиг
"""
aperture = 2*1e3 # actual size in micrometers of lens
cell_y = aperture
cell_x = conf.cell_X
cell = mp.Vector3(cell_x, cell_y, 0)  # размер области симуляции
spot_ff = aperture
orders = 5 # the number of radial layers.
stack = 2 # Определяет количество саб-ячеек. При 0 имеет только одну главную ячейку
f = 5*1e3 # focal length 5 mm
Q = 200 # curve detailing. It has almost no effect on the result
w = 1 # 1000 nm. The width of the gap between the curves
wavelength = 4.25
a = 40
LIMIT_PRODUCT = 0.2 # 200 nm
angle = 0
visualize_curve = False
visualize_meep = False

## STEP 2
Инициализация функций получения сечений

In [39]:
"""
Функции
"""

def setup_visualization():
    """Инициализация фигуры для визуализации"""
    global fig, ax
    fig, ax = plt.subplots(figsize=(10, 10))
    # Определяем палитру цветов
    global colors
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
          '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

def plot_curve(xx, yy, X_sections, y0, n):
    """
    Функция для визуализации кривой и точек пересечения
    :param xx: x-координаты кривой
    :param yy: y-координаты кривой
    :param X_sections: точки пересечения с горизонтальной линией
    :param y0: y-координата горизонтальной линии
    :param n: номер кривой (для выбора цвета)
    """

    # Рисуем кривую
    ax.plot(xx, yy, color=colors[n % len(colors)], linewidth=0.5)

    # Выделяем точки пересечения красным
    ax.scatter(X_sections, np.full_like(X_sections, y0), color='red', s=10, zorder=3)
    

def calculate_radii (D, f, wavelength, orders):
    """
    Вычисляет радиусы на поверхности линзы, на которых происходит полный фазовый сдвиг от 0 до 2Pi.

    :param D: Диаметр линзы.
    :param f: Фокусное расстояние линзы.
    :param wavelength: Длина волны света.
    :param orders: Максимальное значение n для расчета радиусов.
    :return: radii: Список радиусов.
    """
    radii = []
    for n in range(1, orders + 1):
        r = np.sqrt((n * wavelength)**2 + 2 * n * wavelength * f)
        if r <= D / 2:  # Проверяем, чтобы радиус не превышал радиус линзы
            radii.append(r)
        else:
            break  # Прекращаем расчет, если радиус превышает размер линзы
    return radii

def catenary_function(x, x0, a):
    """
    Description:
    Используем самую простую функцию описания CC: со центровкой между двумя радиусами
    
    :param x: координата х
    :param x0: центр между радиусами
    :param a: коэффицент кривизны
    """   
    # https://chatgpt.com/share/67e139b7-8b94-800f-beef-0178c329f0d2
    
    return a * np.cosh((x - x0) / a)

def cut_y(xx, yy, y0):
    x_values = []
    for i in range(len(xx) - 1):
        y1, y2 = yy[i], yy[i + 1]
        if (y1 - y0) * (y2 - y0) <= 0:
            x1, x2 = xx[i], xx[i + 1]
            x_intersection = x1 + (y0 - y1) * (x2 - x1) / (y2 - y1)
            x_values.append(x_intersection)
    
    # Сортируем значения по возрастанию
    x_values.sort()

    return np.array(x_values)

def create_2d_curve(orders, stack, rt, Q=200, w=0.1, plot=False):
    """
    :param orders: количество порядков в линзе
    :param stack: количество кривых, которые пересекаются сечением на каждом порядке. Четное число 
    :param rt: лист с радиусами
    :param Q: количество точек на кривую. Default 200 points
    :param w: The width of the gap between the curves. Default 100 nm
    :return: coordinates of X crossections points (as 2D numpy array) and centers
    """    
    list_pure_x = []
    y0 = -0.0001 + 2 * stack * w
       
    if plot:
        setup_visualization()
        ax.axhline(y=y0, color='black', linestyle='--', linewidth=1)
        
    for m in range(orders):
        if m == 0:
            rt0, x0 = 0, 0        
        else: 
            rt0 = rt[m - 1] + 0.01
            x0 = (rt[m] + rt0) / 2  # центры кривых
        
        x = np.linspace(rt0, rt[m] - 0.01, Q)
        y = catenary_function(x, x0, a)  # Предполагается, что `a` определено
        
        # Идём в обратном порядке по стеку (stack, stack-1, ..., 0)
        for n in range(stack, -1, -1):
            ip = np.argmin(y)
            y_shifted = y - y[ip]
            ymax = 0.5 * (np.max(x) - np.min(x))

            y_top = np.clip(y_shifted, -ymax, ymax)
            y_above_zero = np.clip(y_shifted - w, -ymax, ymax)

            xx = np.concatenate([x, x[::-1]])
            yy = np.concatenate([y_top, y_above_zero[::-1]]) + 2 * n * w
            
            x_pure = cut_y(xx, yy, y0)  # Предполагается, что `cut_y` определена
            list_pure_x.append(x_pure)
            
            if plot:
                # Визуализация
                plot_curve(xx, yy, x_pure, y0, n)
    
    # Преобразование в 2D numpy массив с размерностью (orders, stack + 1)
    x_pure = np.array(list_pure_x, dtype=object).reshape(orders, stack + 1)
    
    return x_pure

def calculate_width_and_centers(x_pure):
    """
    Вычисляет разницы в зависимости от количества элементов во вложенных массивах.
    
    Параметры:
    x_pure -- 2D numpy массив, где каждый элемент — массив с 1, 2 или 4 числами
    
    Возвращает:
    numpy массив с результатами (форма сохраняется, содержимое заменяется на разницы)
    """
    # Создаем выходной массив той же формы, но с dtype=object для гибкости
    result_width = np.empty_like(x_pure, dtype=object)
    result_centers = np.empty_like(x_pure, dtype=object)
    
    for i in range(x_pure.shape[0]):
        for j in range(x_pure.shape[1]):
            
            arr = x_pure[i][j]

            if len(arr) == 1 and i == 0 and j == 0:
                result_width[i][j] = np.array([2 * arr[0]]) # Пояснение, далее мы используем ширины которые центрированы. А 0 порядок это по сути урезанная ширина, так как другая сторона не учитывается. И чтобы ячейка не оказалась уменьшенной в 2 раза. Ее надо искусственно расширить.
                result_centers[i][j] = np.array([0])
                
            elif  len(arr) == 2:
                width1 = arr[1] - arr[0]
                result_width[i][j] = np.array([width1])
                center1 = arr[0] + width1 / 2
                result_centers[i][j] = np.array([center1])
                
            elif len(arr) == 4:
                width1 = arr[1] - arr[0]
                width2 = arr[3] - arr[2]
                result_width[i][j] = np.array([width1, width2])
                center1 = arr[0] + width1 / 2
                center2 = arr[2] + width2 / 2
                result_centers[i][j] = np.array([center1, center2])
                
            else:
                result_width[i][j] = np.array([0])
                result_centers[i][j] = np.array([0])
    
    return result_centers, result_width

def get_width_and_center(plot=True):
    """
    Description: Супер функция, которая возвращает numpy координаты центров и ширин для передачи в геометрию MEEP
    """
    rt = calculate_radii(aperture, f, wavelength, orders)
    x_pure = create_2d_curve(orders, stack, rt, Q, w, plot=plot)
    centers, width = calculate_width_and_centers(x_pure)
    return centers, width


## STEP 3
Инициализация функций моделирования MEEP

In [44]:
def infill_geometry(geometry):
    centers, width = get_width_and_center(visualize_curve)
    # add sub
    geometry.append(mp.Block(
        material=conf.in_material,
        size=mp.Vector3(conf.in_sub, mp.inf, 0),
        center=mp.Vector3(conf.sub_center.x),
    ))
    # add atoms
    for i in range(orders):
        for j in range(stack):
            for k in range (centers[i, j].shape):
                if width[i][j][k] < LIMIT_PRODUCT: continue
                else:
                    ###---positive infilling---###
                    geometry.append(mp.Block(
                        material=conf.at_material,
                        size=mp.Vector3(x=conf.at_hei, y=width[i][j][k], z=0), 
                        center=mp.Vector3(x=conf.at_center.x, y=centers[i][j][k], z=0)
                    ))
    return geometry       

def setup_simulation(geometry, sources, k_point, plot=True, coef_resolution = 2):
    sim = mp.Simulation(
        cell_size=cell,
        boundary_layers= [mp.PML(thickness=conf.pml, direction=mp.X, side=mp.ALL)],
        geometry=geometry,
        sources=sources,
        k_point=k_point,
        resolution=conf.min_resolution * coef_resolution,
        # symmetries = [mp.Mirror(mp.Y)]
    )
    
    # Adding a near2far area
    near2far_region = sim.add_near2far(
        conf.inc_fcen,  # центральная частота
        conf.inc_df,      # ширина спектра
        conf.inc_nfreq, # дискретность частот
        mp.Near2FarRegion(center=conf.flux_center, size=mp.Vector3(y=spot_ff)),
    )
    
    if plot:
        plt.figure(figsize=(1, 100))
        sim.plot2D(
            output_plane = mp.Volume(center=mp.Vector3(0,0,0), size=mp.Vector3(cell[0], cell[1],0))
        )
        plt.show()
    
    return sim

In [ ]:
def MEEP_simulation():
    geometry = []
    geometry = infill_geometry(geometry)
    sources_obj = sourses.KvectorSource(frequency=conf.inc_fcen, y = cell_y, angle=angle)
    sources = sources_obj.get_sources()
    k_point=sources_obj.get_k_vector()
    sim = setup_simulation(geometry, sources, k_point, visualize_meep)
    conf.run_end_time(sim)